# Model Evaluation

This notebook evaluates the trained object detection model.


In [ ]:
import sys
from pathlib import Path
import torch

# Add parent directory to path
sys.path.append(str(Path().resolve().parent))

from src.evaluation import evaluate_from_checkpoint, evaluate_model
from src.training import create_model
from src.preprocessing import get_data_loaders
from config.config import MODELS_DIR, CHECKPOINTS_DIR, TRAIN_CONFIG, MODEL_CONFIG


## Load Model


In [ ]:
# Choose which model to evaluate
# Options: 'best_model.pth', 'latest_model.pth', or a checkpoint file
model_path = MODELS_DIR / "best_model.pth"

# Alternative: use a checkpoint
# model_path = CHECKPOINTS_DIR / "checkpoint_epoch_50.pth"

print(f"Evaluating model: {model_path}")


## Evaluate Model


In [ ]:
# Evaluate model
results = evaluate_from_checkpoint(
    checkpoint_path=model_path,
    split='val',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)


## Detailed Results


In [ ]:
# Display detailed results
print(f"\nMean Average Precision (mAP): {results['mAP']:.4f}")
print(f"\nPer-class AP:")
for class_id, ap in sorted(results['APs'].items()):
    class_name = results['class_names'][class_id - 1] if class_id > 0 else 'background'
    print(f"  {class_name:20s}: {ap:.4f}")


## Save Results


In [ ]:
import json
from config.config import RESULTS_DIR

# Save results to JSON
results_dict = {
    'mAP': float(results['mAP']),
    'APs': {str(k): float(v) for k, v in results['APs'].items()},
    'class_names': results['class_names']
}

results_file = RESULTS_DIR / "evaluations" / "evaluation_results.json"
results_file.parent.mkdir(parents=True, exist_ok=True)

with open(results_file, 'w') as f:
    json.dump(results_dict, f, indent=2)

print(f"Results saved to {results_file}")
